In [ ]:
# ============================================================
# ALL CLASSIFICATION ALGORITHMS - COMPLETE ML WORKFLOW
# ============================================================

# -------------------------
# 1. IMPORT LIBRARIES
# -------------------------

import numpy as np
import pandas as pd

from sklearn.datasets import make_classification
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder

from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import KNeighborsClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.svm import SVC
from sklearn.naive_bayes import GaussianNB
from sklearn.ensemble import (
    RandomForestClassifier,
    AdaBoostClassifier,
    GradientBoostingClassifier,
    VotingClassifier
)

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score
)

# XGBoost
from xgboost import XGBClassifier


# ============================================================
# 2. CREATE RANDOM DATASET
# ============================================================

X, y = make_classification(
    n_samples=1000,
    n_features=6,
    n_informative=4,
    n_redundant=0,
    random_state=42
)

X = pd.DataFrame(
    X,
    columns=[
        'age',
        'income',
        'tenure',
        'score',
        'balance',
        'visits'
    ]
)

# Add categorical column
X['gender'] = np.random.choice(
    ['Male', 'Female'],
    size=1000
)

# Add missing values
X.loc[10, 'age'] = np.nan
X.loc[20, 'income'] = np.nan
X.loc[30, 'gender'] = np.nan


# ============================================================
# 3. TRAIN / TEST SPLIT
# ============================================================

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)


# ============================================================
# 4. DEFINE COLUMNS
# ============================================================

num_cols = [
    'age',
    'income',
    'tenure',
    'score',
    'balance',
    'visits'
]

cat_cols = ['gender']


# ============================================================
# 5. PREPROCESSING
# ============================================================

# Numerical preprocessing
num_pipeline = Pipeline([
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', StandardScaler())
])

# Categorical preprocessing
cat_pipeline = Pipeline([
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('encoder', OneHotEncoder(handle_unknown='ignore'))
])

# Combine both
preprocessor = ColumnTransformer([
    ('num', num_pipeline, num_cols),
    ('cat', cat_pipeline, cat_cols)
])


# ============================================================
# 6. EVALUATION FUNCTION
# ============================================================

def evaluate_model(name, model):

    model.fit(X_train, y_train)

    y_pred = model.predict(X_test)
    y_prob = model.predict_proba(X_test)[:, 1]

    print("\n" + "=" * 50)
    print(name)
    print("=" * 50)

    print(f"Accuracy : {accuracy_score(y_test, y_pred):.4f}")
    print(f"Precision: {precision_score(y_test, y_pred):.4f}")
    print(f"Recall   : {recall_score(y_test, y_pred):.4f}")
    print(f"F1 Score : {f1_score(y_test, y_pred):.4f}")
    print(f"ROC-AUC  : {roc_auc_score(y_test, y_prob):.4f}")


# ============================================================
# 7. LOGISTIC REGRESSION
# ============================================================

logistic_model = Pipeline([
    ('preprocessing', preprocessor),
    ('model', LogisticRegression(max_iter=1000))
])

evaluate_model(
    "Logistic Regression",
    logistic_model
)


# ============================================================
# 8. KNN
# ============================================================

knn_model = Pipeline([
    ('preprocessing', preprocessor),
    ('model', KNeighborsClassifier(n_neighbors=5))
])

evaluate_model(
    "KNN",
    knn_model
)


# ============================================================
# 9. DECISION TREE
# ============================================================

decision_tree_model = Pipeline([
    ('preprocessing', preprocessor),
    ('model', DecisionTreeClassifier(random_state=42))
])

evaluate_model(
    "Decision Tree",
    decision_tree_model
)


# ============================================================
# 10. SVM
# ============================================================

svm_model = Pipeline([
    ('preprocessing', preprocessor),
    ('model', SVC(
        probability=True,
        random_state=42
    ))
])

evaluate_model(
    "SVM",
    svm_model
)


# ============================================================
# 11. NAIVE BAYES
# ============================================================

# GaussianNB requires dense input
nb_preprocessor = ColumnTransformer([
    ('num', num_pipeline, num_cols),

    ('cat', Pipeline([
        ('imputer', SimpleImputer(strategy='most_frequent')),
        ('encoder', OneHotEncoder(
            handle_unknown='ignore',
            sparse_output=False
        ))
    ]), cat_cols)
])

naive_bayes_model = Pipeline([
    ('preprocessing', nb_preprocessor),
    ('model', GaussianNB())
])

evaluate_model(
    "Naive Bayes",
    naive_bayes_model
)


# ============================================================
# 12. RANDOM FOREST
# ============================================================

random_forest_model = Pipeline([
    ('preprocessing', preprocessor),
    ('model', RandomForestClassifier(
        n_estimators=100,
        random_state=42
    ))
])

evaluate_model(
    "Random Forest",
    random_forest_model
)


# ============================================================
# 13. ADABOOST
# ============================================================

adaboost_model = Pipeline([
    ('preprocessing', preprocessor),
    ('model', AdaBoostClassifier(
        random_state=42
    ))
])

evaluate_model(
    "AdaBoost",
    adaboost_model
)


# ============================================================
# 14. GRADIENT BOOSTING
# ============================================================

gradient_boosting_model = Pipeline([
    ('preprocessing', preprocessor),
    ('model', GradientBoostingClassifier(
        random_state=42
    ))
])

evaluate_model(
    "Gradient Boosting",
    gradient_boosting_model
)


# ============================================================
# 15. XGBOOST
# ============================================================

xgboost_model = Pipeline([
    ('preprocessing', preprocessor),
    ('model', XGBClassifier(
        n_estimators=100,
        random_state=42,
        eval_metric='logloss'
    ))
])

evaluate_model(
    "XGBoost",
    xgboost_model
)


# ============================================================
# 16. VOTING CLASSIFIER
# ============================================================

voting_model = Pipeline([
    ('preprocessing', preprocessor),

    ('model', VotingClassifier(
        estimators=[
            ('lr', LogisticRegression(max_iter=1000)),
            ('knn', KNeighborsClassifier(n_neighbors=5)),
            ('dt', DecisionTreeClassifier(random_state=42))
        ],
        voting='soft'
    ))
])

evaluate_model(
    "Voting Classifier",
    voting_model
)


# ============================================================
# DONE
# ============================================================

print("\n" + "=" * 50)
print("ALL MODELS TRAINED AND EVALUATED")
print("=" * 50)